In [1]:
!pip -q install datasets scikit-learn pandas pyarrow

In [2]:
# ============================================================
# Classical baselines on AG News / IMDB / SST-2
# Models:
#   1) TF-IDF + Logistic Regression
#   2) TF-IDF + SVM
# Save all logs/results to Google Drive/MyDrive
# ============================================================

import os
import gc
import json
import time
import random
import warnings
import numpy as np
import pandas as pd

from datasets import load_dataset
from google.colab import drive

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

warnings.filterwarnings("ignore")

# =========================
# 1) Mount Google Drive
# =========================
drive.mount("/content/drive", force_remount=True)

# =========================
# 2) Config
# =========================
CONFIG = {
    "seed": 42,
    "output_root": "/content/drive/MyDrive/classical_baselines_textcls",
    "experiment_name": "tfidf_lr_svm_agnews_imdb_sst2",
    "tfidf_max_features": 50000,
    "ngram_range": (1, 2),
    "min_df": 2,
    "max_iter_lr": 2000,
    "C_lr": 1.0,
    "C_svm": 1.0,
}

EXPERIMENT_DIR = os.path.join(CONFIG["output_root"], CONFIG["experiment_name"])
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

RESULTS_JSON = os.path.join(EXPERIMENT_DIR, "classical_results.json")
RESULTS_CSV = os.path.join(EXPERIMENT_DIR, "classical_results.csv")
CONFIG_JSON = os.path.join(EXPERIMENT_DIR, "run_config.json")

# =========================
# 3) Utilities
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(CONFIG["seed"])

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def evaluate_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return float(acc), float(macro_f1)

def load_text_dataset(dataset_name):
    """
    Returns:
        X_train, y_train, X_test, y_test
    """

    if dataset_name == "ag_news":
        ds = load_dataset("ag_news")
        X_train = ds["train"]["text"]
        y_train = ds["train"]["label"]
        X_test = ds["test"]["text"]
        y_test = ds["test"]["label"]

    elif dataset_name == "imdb":
        ds = load_dataset("imdb")
        X_train = ds["train"]["text"]
        y_train = ds["train"]["label"]
        X_test = ds["test"]["text"]
        y_test = ds["test"]["label"]

    elif dataset_name == "sst2":
        ds = load_dataset("glue", "sst2")
        X_train = ds["train"]["sentence"]
        y_train = ds["train"]["label"]
        X_test = ds["validation"]["sentence"]   # SST-2 thường dùng validation để eval
        y_test = ds["validation"]["label"]

    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")

    return X_train, y_train, X_test, y_test

def build_pipeline(model_type):
    if model_type == "logistic_regression":
        clf = LogisticRegression(
            C=CONFIG["C_lr"],
            max_iter=CONFIG["max_iter_lr"],
            random_state=CONFIG["seed"],
            n_jobs=-1,
        )
        setting_name = "TF-IDF + Logistic Regression"

    elif model_type == "svm":
        clf = LinearSVC(
            C=CONFIG["C_svm"],
            random_state=CONFIG["seed"],
        )
        setting_name = "TF-IDF + SVM"

    else:
        raise ValueError(f"Unsupported model type: {model_type}")

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=CONFIG["tfidf_max_features"],
            ngram_range=CONFIG["ngram_range"],
            min_df=CONFIG["min_df"],
            strip_accents="unicode",
            lowercase=True,
            sublinear_tf=True,
        )),
        ("clf", clf)
    ])

    return pipeline, setting_name

def run_experiment(dataset_name, model_type):
    print("=" * 100)
    print(f"Dataset: {dataset_name} | Model: {model_type}")
    print("=" * 100)

    X_train, y_train, X_test, y_test = load_text_dataset(dataset_name)

    pipeline, setting_name = build_pipeline(model_type)

    # Train
    train_start = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - train_start

    # Inference
    infer_start = time.time()
    y_pred = pipeline.predict(X_test)
    inference_time = time.time() - infer_start

    accuracy, macro_f1 = evaluate_predictions(y_test, y_pred)

    result = {
        "dataset_name": dataset_name,
        "model_name": model_type,
        "setting": setting_name,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "train_time": float(train_time),
        "inference_time": float(inference_time),
        "num_train_samples": int(len(X_train)),
        "num_test_samples": int(len(X_test)),
        "tfidf_max_features": int(CONFIG["tfidf_max_features"]),
    }

    print(json.dumps(result, indent=2))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=4))

    del pipeline, X_train, y_train, X_test, y_test, y_pred
    gc.collect()

    return result

# =========================
# 4) Run all experiments
# =========================
save_json(CONFIG, CONFIG_JSON)

datasets_to_run = ["ag_news", "imdb", "sst2"]
models_to_run = ["logistic_regression", "svm"]

all_results = []

for dataset_name in datasets_to_run:
    for model_type in models_to_run:
        result = run_experiment(dataset_name, model_type)
        all_results.append(result)

        # Save after each run
        save_json(all_results, RESULTS_JSON)
        pd.DataFrame(all_results).to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")

# =========================
# 5) Final summary
# =========================
results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
save_json(all_results, RESULTS_JSON)

print("\n" + "=" * 100)
print("ALL EXPERIMENTS FINISHED")
print("=" * 100)
print(f"Experiment dir : {EXPERIMENT_DIR}")
print(f"Results JSON   : {RESULTS_JSON}")
print(f"Results CSV    : {RESULTS_CSV}")
print(f"Config JSON    : {CONFIG_JSON}")

print("\nGenerated files:")
print(f"- {RESULTS_JSON}")
print(f"- {RESULTS_CSV}")
print(f"- {CONFIG_JSON}")

print("\nFinal results table:")
display(results_df)

Mounted at /content/drive
Dataset: ag_news | Model: logistic_regression


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

{
  "dataset_name": "sst2",
  "model_name": "logistic_regression",
  "setting": "TF-IDF + Logistic Regression",
  "accuracy": 0.8096330275229358,
  "macro_f1": 0.8089001779613132,
  "train_time": 6.30299711227417,
  "inference_time": 0.04667091369628906,
  "num_train_samples": 67349,
  "num_test_samples": 872,
  "tfidf_max_features": 50000
}

Classification report:
              precision    recall  f1-score   support

           0     0.8359    0.7617    0.7971       428
           1     0.7884    0.8559    0.8207       444

    accuracy                         0.8096       872
   macro avg     0.8121    0.8088    0.8089       872
weighted avg     0.8117    0.8096    0.8091       872

Dataset: sst2 | Model: svm
{
  "dataset_name": "sst2",
  "model_name": "svm",
  "setting": "TF-IDF + SVM",
  "accuracy": 0.8165137614678899,
  "macro_f1": 0.8160453979705926,
  "train_time": 3.7926278114318848,
  "inference_time": 0.06528353691101074,
  "num_train_samples": 67349,
  "num_test_samples": 8

,dataset_name,model_name,setting,accuracy,macro_f1,train_time,inference_time,num_train_samples,num_test_samples,tfidf_max_features
0,ag_news,logistic_regression,TF-IDF + Logistic Regression,0.919737,0.919582,32.061345,0.635785,120000,7600,50000
1,ag_news,svm,TF-IDF + SVM,0.922368,0.922268,28.517621,0.698178,120000,7600,50000
2,imdb,logistic_regression,TF-IDF + Logistic Regression,0.900920,0.900918,19.102781,10.647198,25000,25000,50000
3,imdb,svm,TF-IDF + SVM,0.898000,0.898000,18.463616,10.376552,25000,25000,50000
4,sst2,logistic_regression,TF-IDF + Logistic Regression,0.809633,0.808900,6.302997,0.046671,67349,872,50000
5,sst2,svm,TF-IDF + SVM,0.816514,0.816045,3.792628,0.065284,67349,872,50000
